# Module 5: Two Way Fixed Effects Done Properly

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Two way fixed effects is the workhorse estimator for panel difference in
differences, and "the TWFE estimate" is spoken of as though it were one
number. It is not. **The linear and the count versions weight the units
differently, and on this panel they are five percentage points apart.**

Neither is biased. They estimate different things.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

KEEP = [a for a in TRAINED if a != "A007"]     # the pre trend violator, Intermediate 8
BASELINE = profile.set_index("agency_id")["pre_program_uof_per_100_arrests"]

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0
d["base"] = d["agency_id"].map(BASELINE)

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated, form=None, outcome="n_uof", offset=None):
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated))
                  & (s["period"] == "phase")).astype(float)
    fo = form or f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase"
    z = smf.glm(fo, s, family=sm.families.Poisson(),
                offset=s["lo"] if offset is None else offset).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z

## 2. The two specifications

In [ ]:
d["rate"] = 100 * d["n_uof"] / d["n_arrests"]
d["settled"] = ((d["agency_id"].isin(KEEP)) & (d["period"] == "after")).astype(float)
d["phase"] = ((d["agency_id"].isin(KEEP)) & (d["period"] == "phase")).astype(float)

zp = smf.glm("n_uof ~ C(agency_id)+C(year_month)+settled+phase", d,
             family=sm.families.Poisson(), offset=d["lo"]).fit()
zo = smf.ols("rate ~ C(agency_id)+C(year_month)+settled+phase", d).fit()
base = d[(d["agency_id"].isin(KEEP)) & (d["period"] == "before")]["rate"].mean()

lo, hi = zp.conf_int().loc["settled"]
print(f"  Poisson with an offset   {pct(zp.params['settled']):+6.2f}%  "
      f"[{pct(lo):+.1f}, {pct(hi):+.1f}]")
lo, hi = zo.conf_int().loc["settled"]
print(f"  linear on the rate       {zo.params['settled']:+6.3f} rate points  "
      f"[{lo:+.3f}, {hi:+.3f}]")
print(f"                           which is {100 * zo.params['settled'] / base:+.1f}% "
      f"of the pre period mean of {base:.2f}")
print(f"\n  the truth: {TRUTH:+.2f}%")

**Five percentage points apart**, and both are difference in differences with
agency and month fixed effects on the same 964 rows.

## 3. Where the difference comes from

The Poisson model with an offset is fitted to counts, so an agency month
contributes in proportion to the incidents it contains. The linear model on a
rate treats every agency month as one observation.

In [ ]:
post = d[(d["agency_id"].isin(KEEP)) & (d["period"] == "after")]
inc = post.groupby("agency_id")["n_uof"].sum()
inc = 100 * inc / inc.sum()
mon = post.groupby("agency_id").size()
mon = 100 * mon / mon.sum()
per = {a: fit(d[d["agency_id"].isin([a] + COMPARISON)], [a])[0] for a in KEEP}

pd.DataFrame([{"agency": NAME[a].split()[0],
               "share of incidents": f"{inc[a]:.0f}%",
               "share of agency months": f"{mon[a]:.0f}%",
               "its own estimate": f"{per[a]:+.1f}%"} for a in KEEP]).set_index("agency")

Stonewick carries **60 percent of the incidents and 25 percent of the months**,
and its own estimate is the smallest of the four. The Poisson model therefore
lands nearer to Stonewick's number and the linear model nearer to the simple
average, which the small noisy agencies pull down.

**This is the estimand question from [Module 1](Module_01_Potential_Outcomes_And_Estimands.ipynb)
arriving through the back door.** Choosing a link function chooses a
weighting, usually without anyone noticing.

## 4. Which to use

| Situation | Estimator |
|---|---|
| The outcome is a count with an exposure | Poisson with a log offset |
| The question is about incidents statewide | Poisson, or linear weighted by exposure |
| The question is about the typical agency | linear, or Poisson with agency weights |
| Zeros are common | Poisson; the log of a rate drops them |
| Unit sizes are similar | it does not matter, and say so |

**And report both when they differ.** A five point gap between two defensible
estimators is information about how much the answer depends on weighting, and
suppressing it is a choice the reader cannot see.

In [ ]:
w = post.groupby("agency_id")["n_arrests"].sum()
zw = smf.wls("rate ~ C(agency_id)+C(year_month)+settled+phase", d,
             weights=d["n_arrests"]).fit()
print(f"  linear, unweighted        "
      f"{100 * zo.params['settled'] / base:+.1f}%")
print(f"  linear, weighted by arrests "
      f"{100 * zw.params['settled'] / base:+.1f}%")
print(f"  Poisson with an offset    {pct(zp.params['settled']):+.1f}%")
print(f"  the truth                 {TRUTH:+.1f}%")

Weighting the linear model by exposure moves it from 18.2 to 15.1 percent,
closing more than half the gap to the Poisson answer. **The weights were most
of the difference, and the remainder is the functional form: a linear model in
the rate and a log link do not impose the same shape.**

Both facts belong in a report that gives one number.

## Exercise

The two estimators agree when the units are similar in size. Test that by
dropping the largest and smallest treated agencies.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    rows = []
    for label, ids in [("all four treated agencies", KEEP),
                       ("the two middle sized only", ["A002", "A004"])]:
        s = d[d["agency_id"].isin(ids + COMPARISON)].copy()
        s["settled"] = ((s["agency_id"].isin(ids)) & (s["period"] == "after")).astype(float)
        s["phase"] = ((s["agency_id"].isin(ids)) & (s["period"] == "phase")).astype(float)
        p_ = smf.glm("n_uof ~ C(agency_id)+C(year_month)+settled+phase", s,
                     family=sm.families.Poisson(), offset=s["lo"]).fit()
        o_ = smf.ols("rate ~ C(agency_id)+C(year_month)+settled+phase", s).fit()
        b_ = s[(s["agency_id"].isin(ids)) & (s["period"] == "before")]["rate"].mean()
        rows.append({"treated agencies": label,
                     "Poisson": f"{pct(p_.params['settled']):+.1f}%",
                     "linear": f"{100 * o_.params['settled'] / b_:+.1f}%",
                     "gap": round(abs(pct(p_.params["settled"])
                                      - 100 * o_.params["settled"] / b_), 1)})
    display(pd.DataFrame(rows).set_index("treated agencies"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

With only the two middle sized agencies the two estimators come much closer
together, because the weights they imply are now nearly the same.

**That is the diagnostic worth taking away.** If the Poisson and linear
versions of the same design agree, the weighting is not doing much and either
can be reported. If they disagree, the disagreement is a measurement of how
much the answer depends on which units count for how much, and it belongs in
the report rather than in a choice made silently at the point of writing the
formula.

The check costs one extra line and it is the only way to find out.

</details>

---

**Next:** [Module 6: Event Studies and Pre Trend Testing](Module_06_Event_Studies_And_Pre_Trend_Testing.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*